# Lesson 12B: Multi-Agent Reinforcement Learning - Practical

## Introduction

In Lesson 12A we covered the theory: Markov games, Nash equilibria, credit assignment, independent learners, and CTDE. Here we build on real multi-agent environments via **PettingZoo**:

1. **Competitive**: Rock-Paper-Scissors (`classic.rps_v2`) with independent Q-learning
2. **Cooperative**: Pursuit-evasion (`sisl.pursuit_v4`) with a parameter-shared DQN-lite agent, trained via PettingZoo's parallel API
3. **Stable-Baselines3 reproduction**: bridge the same cooperative task into a genuine Gym-compatible `VecEnv` via SuperSuit and train PPO on it

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import deque, Counter

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cpu')  # small MLPs -- CPU is plenty and avoids PettingZoo/CUDA obs dtype friction

## Part 1: The PettingZoo API

PettingZoo exposes two interfaces to the same underlying environment:

- **AEC** (Agent-Environment-Cycle): agents act **one at a time**, in turn, via `env.agent_iter()`. This is the natural fit for turn-based games (Rock-Paper-Scissors, Chess).
- **Parallel**: every agent submits an action **simultaneously** each step, via a single `env.step(actions_dict)`. This is the natural fit for physically-simultaneous multi-agent worlds (pursuit, navigation).

Every agent's individual `observation_space(agent)` / `action_space(agent)` is a standard `gymnasium.spaces` object — PettingZoo is a multi-agent generalization of the Gymnasium API, not a replacement for it.

## Part 2: Competitive Game - Independent Q-Learning on Rock-Paper-Scissors

Two independent Q-learners (one per player), using the AEC turn-based loop. Since RPS has **no pure-strategy Nash equilibrium** (12A, Part 2), we don't expect convergence to a fixed joint action — the interesting result is whether the players' strategies keep cycling instead of settling down.

In [ ]:
from pettingzoo.classic import rps_v2


class IndependentQLearner:
    """Same algorithm as 12A Part 3, applied to a real PettingZoo environment."""

    def __init__(self, n_actions, alpha=0.1, epsilon=0.2):
        self.n_actions = n_actions
        self.alpha, self.epsilon = alpha, epsilon
        self.Q = np.zeros(n_actions)

    def act(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.Q))

    def update(self, action, reward):
        self.Q[action] += self.alpha * (reward - self.Q[action])


rps_env = rps_v2.env(num_actions=3, max_cycles=1)
rps_agents = {name: IndependentQLearner(3) for name in ['player_0', 'player_1']}
action_history = {name: [] for name in rps_agents}

n_rounds = 3000
for _ in range(n_rounds):
    rps_env.reset()
    episode_reward = {name: 0.0 for name in rps_agents}
    last_action = {}
    for agent in rps_env.agent_iter():
        obs, reward, terminated, truncated, info = rps_env.last()
        episode_reward[agent] += reward
        if terminated or truncated:
            rps_env.step(None)
            continue
        action = rps_agents[agent].act()
        last_action[agent] = action
        action_history[agent].append(action)
        rps_env.step(action)
    for agent in rps_agents:
        rps_agents[agent].update(last_action[agent], episode_reward[agent])

print("Action counts, last 500 rounds (0=Rock, 1=Paper, 2=Scissors):")
for agent in rps_agents:
    print(f"  {agent}: {dict(Counter(action_history[agent][-500:]))}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
window = 100
for agent in rps_agents:
    rock_rate = np.convolve(np.array(action_history[agent]) == 0, np.ones(window) / window, mode='valid')
    ax.plot(rock_rate, label=f'{agent} P(Rock)')
ax.axhline(1 / 3, color='gray', linestyle='--', label='Nash mixed-strategy (1/3)')
ax.set_xlabel('Round')
ax.set_ylabel(f'P(Rock), window={window}')
ax.set_title('Independent Q-learners drift around, rather than converge to, the Nash mixture')
ax.legend()
plt.tight_layout()
plt.show()

## Part 3: Cooperative Task - Parameter-Shared DQN on Pursuit

`sisl.pursuit_v4`: pursuers cooperate to surround and catch evaders on a grid, each seeing only a local `7x7` window. We use **parameter sharing** — one Q-network, used by every pursuer, conditioned only on that pursuer's own local observation. This is the cheapest possible CTDE-adjacent trick: agents remain decentralized at execution (each acts on local information only) while training pools every agent's experience into a single shared network.

We loosen the catch condition (`n_catch=1`, `surround=False`) so catches happen often enough, even early in training, to produce a usable learning signal within a notebook-scale training budget.

In [ ]:
from pettingzoo.sisl import pursuit_v4


class SharedQNet(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


def flatten_obs(obs):
    return obs.flatten().astype(np.float32)


pursuit_env = pursuit_v4.parallel_env(
    n_pursuers=2, n_evaders=2, x_size=10, y_size=10, max_cycles=100, n_catch=1, surround=False,
)
obs, info = pursuit_env.reset(seed=0)
obs_dim = int(np.prod(pursuit_env.observation_space(pursuit_env.agents[0]).shape))
n_actions = pursuit_env.action_space(pursuit_env.agents[0]).n

q_net = SharedQNet(obs_dim, n_actions).to(device)
target_net = SharedQNet(obs_dim, n_actions).to(device)
target_net.load_state_dict(q_net.state_dict())
optimizer = optim.Adam(q_net.parameters(), lr=5e-4)
replay_buffer = deque(maxlen=20000)
gamma = 0.95
batch_size = 64

n_episodes = 60
episode_returns = []
step_count = 0
for episode in range(n_episodes):
    epsilon = max(0.05, 1.0 - episode / (0.6 * n_episodes))
    obs, info = pursuit_env.reset(seed=episode)
    total_return = 0.0
    while pursuit_env.agents:
        actions, obs_flat = {}, {a: flatten_obs(obs[a]) for a in pursuit_env.agents}
        for agent in pursuit_env.agents:
            if np.random.rand() < epsilon:
                actions[agent] = pursuit_env.action_space(agent).sample()
            else:
                with torch.no_grad():
                    actions[agent] = int(torch.argmax(q_net(torch.as_tensor(obs_flat[agent]))).item())
        next_obs, rewards, terminations, truncations, infos = pursuit_env.step(actions)
        for agent in pursuit_env.agents:
            reward = rewards.get(agent, 0.0)
            total_return += reward
            done = terminations.get(agent, False) or truncations.get(agent, False)
            next_flat = flatten_obs(next_obs[agent]) if agent in next_obs else obs_flat[agent]
            replay_buffer.append((obs_flat[agent], actions[agent], reward, next_flat, done))
        obs = next_obs
        step_count += 1

        if len(replay_buffer) >= batch_size:
            batch = [replay_buffer[i] for i in np.random.randint(0, len(replay_buffer), batch_size)]
            s = torch.as_tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
            a = torch.as_tensor([b[1] for b in batch], dtype=torch.long)
            r = torch.as_tensor([b[2] for b in batch], dtype=torch.float32)
            s_next = torch.as_tensor(np.array([b[3] for b in batch]), dtype=torch.float32)
            d = torch.as_tensor([b[4] for b in batch], dtype=torch.float32)
            q_values = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                targets = r + gamma * (1 - d) * torch.max(target_net(s_next), dim=1).values
            loss = nn.functional.mse_loss(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if step_count % 200 == 0:
            target_net.load_state_dict(q_net.state_dict())

    episode_returns.append(total_return)

print(f"First 10 episodes mean team return: {np.mean(episode_returns[:10]):.1f}")
print(f"Last 10 episodes mean team return:  {np.mean(episode_returns[-10:]):.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
window = 10
smoothed = np.convolve(episode_returns, np.ones(window) / window, mode='valid')
ax.plot(episode_returns, alpha=0.3, label='raw')
ax.plot(smoothed, linewidth=2, label=f'smoothed (window={window})')
ax.set_xlabel('Episode')
ax.set_ylabel('Team return')
ax.set_title('Parameter-shared DQN-lite learns to cooperate on Pursuit')
ax.legend()
plt.tight_layout()
plt.show()

## Part 4: Stable-Baselines3 Reproduction

SB3 agents are single-policy, single-environment learners — they don't natively understand a multi-agent PettingZoo env. **SuperSuit** bridges the gap the same way our shared network did conceptually: it wraps the parallel PettingZoo env into a single `VecEnv` where each agent is treated as one more (Gym-API-compatible) sub-environment, so a single PPO policy trains across all of them with parameter sharing built in.

In [ ]:
import supersuit as ss
from stable_baselines3 import PPO

sb3_pursuit_env = pursuit_v4.parallel_env(
    n_pursuers=2, n_evaders=2, x_size=10, y_size=10, max_cycles=100, n_catch=1, surround=False,
)
sb3_pursuit_env = ss.pettingzoo_env_to_vec_env_v1(sb3_pursuit_env)
sb3_pursuit_env = ss.concat_vec_envs_v1(sb3_pursuit_env, 1, num_cpus=1, base_class='stable_baselines3')

sb3_model = PPO('MlpPolicy', sb3_pursuit_env, verbose=0, n_steps=256, batch_size=64, gamma=0.95, ent_coef=0.01)
sb3_model.learn(total_timesteps=60000)

eval_env = pursuit_v4.parallel_env(
    n_pursuers=2, n_evaders=2, x_size=10, y_size=10, max_cycles=100, n_catch=1, surround=False,
)
sb3_returns, random_returns = [], []
for ep in range(8):
    obs, info = eval_env.reset(seed=100 + ep)
    total = 0.0
    while eval_env.agents:
        actions = {a: sb3_model.predict(obs[a], deterministic=True)[0] for a in eval_env.agents}
        obs, rewards, terminations, truncations, infos = eval_env.step(actions)
        total += sum(rewards.values())
    sb3_returns.append(total)

for ep in range(8):
    obs, info = eval_env.reset(seed=200 + ep)
    total = 0.0
    while eval_env.agents:
        actions = {a: eval_env.action_space(a).sample() for a in eval_env.agents}
        obs, rewards, terminations, truncations, infos = eval_env.step(actions)
        total += sum(rewards.values())
    random_returns.append(total)

print(f"Random policy      - mean team return: {np.mean(random_returns):.1f}")
print(f"From-scratch shared DQN (60 episodes) - mean team return: {np.mean(episode_returns[-10:]):.1f}")
print(f"SB3 PPO (60,000 timesteps) - mean team return: {np.mean(sb3_returns):.1f}")

## Key Takeaways

1. **AEC vs. Parallel**: PettingZoo's two interfaces match the underlying game's structure — turn-based (AEC) vs. simultaneous-move (Parallel) — both built on standard `gymnasium.spaces`
2. **Independent Q-learning on a zero-sum game** doesn't converge to the mixed-strategy Nash equilibrium — it drifts, echoing 12A's theoretical point that IQL ignores non-stationarity
3. **Parameter sharing** is the cheapest practical CTDE-adjacent trick: one network trained on every agent's experience, each agent still acting on local information only
4. **SuperSuit** bridges PettingZoo's multi-agent API to SB3's single-policy `VecEnv` interface by treating each agent as another vectorized sub-environment
5. Both learned approaches clearly beat the random baseline; the from-scratch DQN-lite and SB3 PPO land in a comparable range at this training budget (their relative order isn't stable across seeds) — PPO is on-policy and typically needs more samples than a replay-buffer method to pull ahead on a sparse-reward task like this one